In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import sqlite3
from tqdm.notebook import tqdm
import rapidfuzz
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt

from sys import path as sys_path
project_root = Path.cwd().parent.parent
if project_root not in sys_path:
    sys_path.append(str(project_root))

In [2]:
data_dir = project_root/"data/"
outputs_dir = project_root/"outputs/"
input_dir = outputs_dir/"toponym_cols/parishes"

## Hiski

Files with occupations

In [4]:
files = (
    data_dir/"hiski-2026-09-22-cnv/ammhaud.cnv",
    data_dir/"hiski-2026-09-22-cnv/ammkasta.cnv",
    data_dir/"hiski-2026-09-22-cnv/ammkasti.cnv",
    data_dir/"hiski-2026-09-22-cnv/ammkastk.cnv",
    data_dir/"hiski-2026-09-22-cnv/ammmuutt.cnv",
    data_dir/"hiski-2026-09-22-cnv/ammom.cnv",
    data_dir/"hiski-2026-09-22-cnv/ammvihi.cnv",
    data_dir/"hiski-2026-09-22-cnv/ammvihv.cnv"
)

Read groups from files

In [5]:
groups = []
for file in files:
    with open(file, "r") as fp:
        groups += [row.strip().split("@") for row in fp.readlines()]

Get keys for groups

In [ ]:
keys = [group[0] for group in groups]
duplicate_keys = [k for k in keys if keys.count(k) > 1]

Inspect and resolve groups with conflicting keys

In [18]:
sorted(filter(lambda x: x[0] in duplicate_keys, groups))

[['Bondedottren'],
 ['Bondedottren', 'bd.dtr', 'bd.dr', 'Bondensdotter'],
 ['Bonden', 'B.den', 'B.de', 'Bond', 'Bd', 'B:n'],
 ['Bonden', 'Bden', 'bd:'],
 ['Bonden',
  'Talon isäntä, leski',
  'Talonisäntä, Leski',
  'Taloll.',
  'Taloll. leski',
  'Talon isäntä',
  'Talollinen leskimies',
  'Talollinen nuorimies',
  'Talonisäntä leskimies',
  'Talonp.',
  'Talonp. nuorim.',
  'Talonp. nuorimies',
  'Talonpoika nuorimies',
  'enkl: bd:',
  'bd:',
  'uk: bd:'],
 ['Bondesonen'],
 ['Bondesonen',
  'Bd:s.',
  'Bdeson',
  'Bdson',
  'Bs:n',
  'Bson',
  'Bonde sonen',
  'Talonpoika'],
 ['Bondesonen', 'bd.son', 'b.son', 'b.s.', 'bd.s.'],
 ['Dreng', 'Drengen'],
 ['Dreng',
  'Drengen',
  'Drängen',
  'Drengen enklingen',
  'Renki',
  'trenki',
  'trenki ja nuorimies',
  'trenki, nuorimies',
  'Trenki leskim.',
  'Trenki nuorim.'],
 ['Klockare', 'Clåck:', 'Klock.', 'Klockaren', 'Klockn', 'Klåck.'],
 ['Klockare', 'klock:'],
 ['Nämndeman',
  'Nämd.',
  'Nämdm:',
  'Nämnd.',
  'Nämndem.',
  'Nämdem.

The listed groups are same and can be combined.

In [ ]:
occupation_group_dict = {}
for occupation_group in groups:
    if occupation_group_dict.get(occupation_group[0]) == None:
        occupation_group_dict[occupation_group[0]] = set(occupation_group)
    else:
        occupation_group_dict[occupation_group[0]].update(
            occupation_group
        )

In [31]:
with open(outputs_dir/"occupations/occupation_groups.json", "w") as fp:
    json.dump(
        {
            key: list(group)
            for key, group in sorted(occupation_group_dict.items())
        },
        fp=fp,
        ensure_ascii=False,
        indent=4
    )